## Reading Amazon Reviews Final Project

In [1]:
import sys
print(sys.version)
print(spark.version)

3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:38:13) [GCC 12.3.0]
3.5.1


In [2]:
import os
import subprocess
import datetime
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [3]:
gcs_folder = 'gs://msca-bdp-data-open/final_project_reviews'

#### Check data size in GCS

In [4]:
cmd = 'gsutil du -s -h ' + gcs_folder

p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
for line in p.stdout.readlines():
    print (f'Total directory size: {line}')
    
retval = p.wait() # Wait for the child process to terminate.

Total directory size: 75.75 GiB    gs://msca-bdp-data-open/final_project_reviews



### Read Amazon reviews and meta data from GCS

#### ALL Reviews


In [5]:
%%time   
    
df_reviews = spark.read.parquet(os.path.join(gcs_folder, 'reviews_parquet'))
print(f'Records read from dataframe *reviews*: {df_reviews.count():,.0f}')

Records read from dataframe *reviews*: 64,679,785
CPU times: user 22.4 ms, sys: 14 ms, total: 36.3 ms
Wall time: 13.3 s


In [6]:
display(df_reviews.printSchema())

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



None

In [8]:
spark.conf.set("spark.sql.repl.eagerEval.enabled",True)

In [9]:
df_reviews.limit(2)

asin,helpful_vote,parent_asin,rating,text,timestamp,title,user_id,verified_purchase
B00HSAOFD8,0,B00HSAOFD8,4.0,Brushes arrived q...,1453819207000,Four Stars,AGWL43EG6H3AO3CN2...,true
B07MGKRMBG,0,B07SPY1GJG,5.0,I have tried just...,1600341347791,Finally a natural...,AGUUIUEN3Y4X54WXM...,true


#### ALL Meta data


In [5]:
# When set to True, Spark will strictly require exact casing, which can help prevent ambiguous or accidental mismatches in large schemas.
#spark.conf.set('spark.sql.caseSensitive', True)

In [7]:
%%time

df_meta = spark.read.parquet(os.path.join(gcs_folder, 'meta_parquet'))
print(f'Records read from dataframe *meta*: {df_meta.count():,.0f}')

Records read from dataframe *meta*: 4,320,533
CPU times: user 12.3 ms, sys: 4.17 ms, total: 16.5 ms
Wall time: 2.74 s


In [10]:
display(df_meta.printSchema())

root
 |-- author: struct (nullable = true)
 |    |-- about: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- avatar: string (nullable = true)
 |    |-- name: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- bought_together: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- price: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- title: string (nullable = true)



None

In [13]:
df_meta.limit(10)

author,average_rating,bought_together,categories,description,main_category,parent_asin,price,rating_number,store,subtitle,title
NULL,5.0,NULL,[Beauty & Persona...,[],All Beauty,B0771WJBHK,NULL,5,Bare Alchemy,NULL,Soothe Serum for ...
NULL,4.0,NULL,[Beauty & Persona...,[],NULL,B09NKNNS57,NULL,17,DISCOVER DEVICE,NULL,DISCOVER Tattoo B...
NULL,4.6,NULL,[Beauty & Persona...,[],All Beauty,B0C6FF5HWL,8.99,137,auroray,NULL,auroray 2200 Pcs ...
NULL,1.0,NULL,[Beauty & Persona...,[],NULL,B09NKSHV3Z,15.69,2,KKNC,NULL,KKNC Chemistry In...
NULL,5.0,NULL,[Beauty & Persona...,[Introduced by th...,All Beauty,B000C213AC,NULL,1,Fine Fragrances,NULL,Aficionado By Fin...
NULL,2.5,NULL,[Beauty & Persona...,[],NULL,B09QJ4Y1V6,NULL,2,LIUYX,NULL,Hot Air Brush Hai...
NULL,2.0,NULL,[Beauty & Persona...,[],All Beauty,B001I4GC8Q,76.03,1,Tom's of Maine,NULL,Tom's of Maine Lo...
NULL,3.0,NULL,[Beauty & Persona...,[],NULL,B002JQKSQO,NULL,1,Barbar,NULL,BARBAR Titanium B...
NULL,3.5,NULL,[Beauty & Persona...,"[Description, Wat...",NULL,B07B3K38QG,11.99,10,FRCOLOR,NULL,Frcolor Shiny Cos...
NULL,4.2,NULL,[Beauty & Persona...,[Neutral Gris Bat...,Grocery,B01MSGFMJU,6.49,29,Grisi,NULL,Grisi Neutral Soa...


In [12]:
import datetime
import pytz

datetime.datetime.now(pytz.timezone('US/Central')).strftime("%a, %d %B %Y %H:%M:%S")

'Wed, 26 November 2025 13:41:37'

In [14]:
from pyspark.sql.functions import col, array_contains

# 1. Select and Filter df_meta aggressively
df_meta_slim = df_meta.select(
    col("parent_asin"),
    col("title").alias("product_title"), # Rename to avoid conflict
    col("store"),
    col("main_category"),
    col("average_rating").alias("product_avg_rating"),
    # Keep 'categories' array for potential granular analysis later
    col("categories"),
    col("price") # Keep price for cleaning/correlation later
).filter(
    # Ensure key identifiers and category are present
    col("parent_asin").isNotNull() & 
    col("product_title").isNotNull() &
    col("main_category").isNotNull()
)

In [15]:
from pyspark.sql.functions import col

# Filter out reviews with poor data quality or missing join key
df_reviews_filtered = df_reviews.filter(
    (col("rating").isNotNull()) & 
    (col("timestamp").isNotNull()) &
    (col("parent_asin").isNotNull()) & # Ensure we can link to metadata
    (col("rating").between(1.0, 5.0))
)

In [16]:
# Perform the efficient lazy join
df_joined = df_reviews_filtered.join(
    df_meta_slim,
    on="parent_asin",
    how="left"
)
# All subsequent analysis will be performed using transformations on df_joined.

In [17]:
df_joined.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- product_title: string (nullable = true)
 |-- store: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- product_avg_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price: string (nullable = true)



In [18]:
from pyspark.sql.functions import count, col

# Define the set of columns that should uniquely identify a review
key_columns = ["user_id", "asin", "timestamp"] 

# 1. Group by the key columns and count occurrences
df_duplicate_counts = df_joined.groupBy(key_columns).agg(
    count("*").alias("count_of_rows")
)

# 2. Filter to find the duplicate rows (where count > 1)
df_duplicates = df_duplicate_counts.filter(col("count_of_rows") > 1)

# 3. ACTION: Get the actual number of true duplicates (safe to collect this small result)
total_duplicates = df_duplicates.count()

print(f"Total number of duplicate row combinations found: {total_duplicates}")

Total number of duplicate row combinations found: 599864


In [19]:
# Define the unique identifier columns for a single review event
key_columns = ["user_id", "asin", "timestamp"] 

# Use dropDuplicates with the subset argument to keep minimal computational emory
df_final = df_joined.dropDuplicates(subset=key_columns)

print(f"Original Row Count: {df_joined.count()}")
print(f"De-duplicated Row Count: {df_final.count()}")

Original Row Count: 64679785


De-duplicated Row Count: 63968473


In [21]:
# Define the base path and the specific folder for your clean data
base_gcs_path = "gs://msca-bdp-students-bucket/notebooks/jiayue1/"
output_folder = "reviews_meta_final_parquet/" 
output_path = base_gcs_path + output_folder

# 1. ACTION: Write the DataFrame to GCS in Parquet format
# This forces the execution of all previous transformations (filter, join, deduplicate).
# Using 'overwrite' is safe here since this is a new output folder.
print(f"Writing data to GCS path: {output_path}")
df_final.write.mode("overwrite").parquet(output_path)

print("Write complete. Starting reload...")

Writing data to GCS path: gs://msca-bdp-students-bucket/notebooks/jiayue1/reviews_meta_final_parquet/


25/11/26 20:23:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Write complete. Starting reload...
